In [1]:
import numpy as np
import pandas as pd
import ast
import nltk
import pickle

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from nltk.stem.porter import PorterStemmer

In [2]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')


In [3]:
print(f"Number of rows and columsn in Movies {movies.shape}")
print(f"Number of rows and columsn in Credits {credits.shape}")

Number of rows and columsn in Movies (4803, 20)
Number of rows and columsn in Credits (4803, 4)


In [4]:
## Since number of rows/entries in both the tables are same, we will merge both table using merge function
movies = movies.merge(credits, on='title')
## after merge, title column will be skipped in final result/table. hence only 23 columns


In [5]:
## Since we are desiging a Movie recommendation model, we will drop some columns which are not required for our analysis

movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [6]:
## check for missing data
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [7]:
movies.dropna(inplace=True)
# after running this cell, check movies.isnull.sum again if the null.na values are removed form dataset

In [8]:
## check for duplicate data
movies.duplicated().sum()

np.int64(0)

In [9]:
# bring columns in correct format 
def convert(obj):
    list=[]
    for i in ast.literal_eval(obj):
        list.append(i['name'])
    return list

def convert3(obj):
    list1=[]
    counter=0
    for i in ast.literal_eval(obj):
        if counter!=3:
            list1.append(i['name'])
            counter+=1
        else:
            break
    return list1


def fetch_director(obj):
    list2=[]
    for i in ast.literal_eval(obj):
        if(i['job']=='Director'):
            list2.append(i['name'])
            break
    return list2

In [10]:
movies['genres'] = movies['genres'].apply(convert)

movies['keywords'] = movies['keywords'].apply(convert)

movies['cast'] = movies['cast'].apply(convert3)

movies['crew'] = movies['crew'].apply(fetch_director)

movies['overview'] =  movies['overview'].apply(lambda x:x.split())

In [11]:
# code is written to remove spaces between two words pointing to same name of the person Sam Worthington,Johnny Depp are names
# of some person. Since a gap between Johnny and Depp can create a confusion for our model, we have to remove these spaces

movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x ])

movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x ])

movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x ])

movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x ])


In [12]:
# create a new column called tags which is a result of concatenation of 5 columns
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew'] 

In [13]:
new_movies = movies[['movie_id','title','tags']]


In [14]:
new_movies['tags'] = new_movies['tags'].apply(lambda x:" ".join(x))
new_movies['tags'] = new_movies['tags'].apply(lambda x:x.lower())

C:\Users\Hp\AppData\Local\Temp\ipykernel_32\3434673521.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_movies['tags'] = new_movies['tags'].apply(lambda x:" ".join(x))
C:\Users\Hp\AppData\Local\Temp\ipykernel_32\3434673521.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_movies['tags'] = new_movies['tags'].apply(lambda x:x.lower())


In [15]:
new_movies['tags'][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

## Text Vectorization

Vectorization of text is a process in Natural Language Processing (NLP) where textual data is transformed into numerical vectors, 
making it suitable for computational analysis and machine learning tasks


In [16]:
countVectorizer = CountVectorizer(max_features=5000,stop_words='english')

In [17]:
vectors = countVectorizer.fit_transform(new_movies['tags']).toarray()

In [18]:
## While vectorization, we have similar words like act, acting, action actions, actionhero activity, activities, actor, actors, actress,
## accident, accidental, accidentally, loved, loving, love it creates new features for our model creating confusion. Hence we perform 
## stemming.


In [19]:
## library for stemming
!pip install nltk

In [20]:

porterStemmer = PorterStemmer()

In [21]:
def stem(text):
    list=[]
    for i in text.split():
        list.append(porterStemmer.stem(i))

    return " ".join(list)

In [22]:
new_movies['tags'] = new_movies['tags'].apply(stem)

C:\Users\Hp\AppData\Local\Temp\ipykernel_32\3229886743.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_movies['tags'] = new_movies['tags'].apply(stem)


## Euclidean Distance vs Cosine Distance

Cosine distance focuses on the angle between vectors, ignoring magnitude.
Euclidean distance focuses on magnitude, measuring the straight-line distance between two points in space.
** Euclidean distance is not a good measure in higher dimensional measure in our case it is 5000 D

In [23]:
# Uncomment and refer below image for more clarity between differences 
## ![image.png](attachment:dc888910-b32f-4990-8893-1a7497d8d807.png)

## Cosine Similarity

In [24]:
similarity = cosine_similarity(vectors)


In [25]:
def recommend(movie):
    # to find the index of any movie
    movie_index = new_movies[new_movies['title'] == movie].index[0]
    distance = similarity[movie_index]
    movies_list = sorted(list(enumerate(distance)),reverse=True,key=lambda x:x[1])[1:6]

    for i in movies_list:
        print(new_movies.iloc[i[0]].title)

In [26]:
recommend('Batman Begins')

The Dark Knight
The Dark Knight Rises
Batman
Batman
Batman & Robin


In [27]:
pickle.dump(new_movies.to_dict(),open('movies_dict.pkl','wb'))

In [28]:
pickle.dump(similarity,open('similarity.pkl','wb'))